In [1]:
import json
import pandas as pd
import numpy as np
from collections import defaultdict
from indicators import get_indicators_for_token
import os
from openpyxl import load_workbook, Workbook
import glob

In [2]:
# Load JSON file
with open(r"C:\Users\vamsh\Downloads\TA MV2\python_server\ML_Training_datasets\CandleData\2dCYTTCQxPsMZeBR2b3x3wmKYytfxsnsEtyy7nwGQzEU.json", "r") as f:
    data = json.load(f)

In [3]:
token_name = data.get("name", "Unknown")
candles = data.get("timeframes", {}).get("5S", [])

if not candles:
    print("No candles found in JSON.")
else:
    # Convert to DataFrame
    df = pd.DataFrame(candles)

    # Convert timestamp to datetime column (do NOT set as index)
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    
    # Sort by timestamp just in case
    df = df.sort_values("timestamp").reset_index(drop=True)

    print(f"Token: {token_name}")


Token: Rizzoween


In [4]:
df.columns

Index(['timestamp', 'open', 'high', 'low', 'close', 'volume'], dtype='object')

In [5]:
df.head(10)

,timestamp,open,high,low,close,volume
0,2025-08-17 05:50:40,129109.899805,129640.049929,129598.210796,129640.049929,6.956520
1,2025-08-17 05:51:15,129640.049929,129007.284960,129007.284960,129007.284960,3.692462
2,2025-08-17 05:53:20,129007.284960,128855.999995,128855.999995,128855.999995,3.714608
3,2025-08-17 05:53:35,128855.999995,128705.453230,128705.453230,128705.453230,21.895836
4,2025-08-17 05:53:50,128705.453230,129329.885600,129329.885600,129329.885600,18.458366
5,2025-08-17 05:54:05,129329.885600,129522.474802,129522.474802,129522.474802,9.646036
6,2025-08-17 05:54:55,129522.474802,128614.792255,128614.792255,128614.792255,49.754711
7,2025-08-17 05:55:00,128614.792255,129057.223121,129057.223121,129057.223121,25.040569
8,2025-08-17 05:55:05,129057.223121,129297.148079,129297.148079,129297.148079,14.859112
9,2025-08-17 05:55:25,129297.148079,130048.601260,130048.601260,130048.601260,111.822600


In [ ]:
def create_forward_label_dataset(df, window_size, step_size, up_threshold, down_threshold, stable_threshold, timeframe, max_horizon):
#     """
#     Label each candle with whether it reaches +threshold% (up) or -threshold% (down)
#     within max_horizon candles. Otherwise mark as stable.

#     Parameters
#     ----------
#     df : DataFrame
#         Must have 'close' and 'timestamp' columns (others optional).
#     threshold : float
#         % move up or down needed to count as extreme (default 10%).
#     timeframe : int
#         Candle duration in seconds (e.g. 15 for 15s).
#     max_horizon : int
#         Max candles to look forward.

#     Returns
#     -------
#     DataFrame with extra columns:
#         - extreme_type : {"up", "down", "stable"}
#         - change_percentage : % move when extreme found (else 0)
#         - time_to_extreme : seconds until extreme (else max_horizon * timeframe)
#     """
#     dataset = []

#     df_len = len(df)
#     start_idx = 0

#     while start_idx + window_size <= df_len:
#         block_df = df.iloc[start_idx:start_idx + window_size].copy()

#         closes = block_df["close"].values
#         N = len(closes)

#         i = 0
#         while i < N:
#             current_close = closes[i]
#             label = None
#             pct = None
#             tte = None
    
#             open_rel  = block_df.iloc[i]['open'] / current_close
#             high_rel  = block_df.iloc[i]['high'] / current_close
#             low_rel   = block_df.iloc[i]['low'] / current_close
#             close_rel = block_df.iloc[i]['close'] / current_close
    
#             extreme_found = False 
#             extreme_j = None       
    
#             up_j = None
#             down_j = None
    
    
#             for j in range(i+40, min(i+max_horizon, N)):
#                 flattened_indicators = None
#                 future_close = closes[j]
#                 pct_change = (future_close - current_close) / current_close * 100
    
#                 if pct_change >= up_threshold:
#                     label = "up"
#                     pct = pct_change
#                     tte = (j - i) * timeframe
#                     if up_j is None:
#                         up_j = j
#                 elif pct_change <= -down_threshold:
#                     label = "down"
#                     pct = pct_change
#                     tte = (j - i) * timeframe
#                     if down_j  is None:
#                         down_j  = j
#                 elif (pct_change >= -stable_threshold and  pct_change <= stable_threshold):
#                     label = "stable"
#                     pct = pct_change
#                     tte = (j - i) * timeframe
                            
#                 if j <= i:
#                     print(f"⚠️ Empty subblock: i={i}, j={j}")
#                     continue  # skip this iteration
                
                
#                 subblock = block_df.iloc[i:j]
#                 indicators = get_indicators_for_token(subblock.to_dict('records'))
#                 flattened_indicators = flatten_and_align_indicators(indicators, len(subblock))
    
#                 # Basic features
#                 flattened_indicators['time_to_extreme'] = tte
#                 flattened_indicators['extreme_type'] = label
#                 flattened_indicators['change_percentage'] = pct_change
                
#                 # Volume features
#                 flattened_indicators['volume_sum'] = subblock['volume'].sum()
#                 flattened_indicators['volume_avg'] = subblock['volume'].mean()
#                 flattened_indicators['volume_rel'] = subblock['volume'].mean() / subblock['volume'].iloc[0]
#                 flattened_indicators['volume_max_rel'] = subblock['volume'].max() / subblock['volume'].mean()
    
#                 # Relative high/low
#                 flattened_indicators['rel_high'] = subblock['high'].max() / current_close
#                 flattened_indicators['rel_low'] = subblock['low'].min() / current_close
#                 flattened_indicators['range_pct'] = (subblock['high'].max() - subblock['low'].min()) / current_close * 100
#                 flattened_indicators['subblock_len'] = len(subblock)
    
#                 # OHLC relative
#                 flattened_indicators['open_rel']  = open_rel
#                 flattened_indicators['high_rel']  = high_rel
#                 flattened_indicators['low_rel']   = low_rel
#                 flattened_indicators['close_rel'] = close_rel
    
#                 # Time features
#                 ts = pd.to_datetime(df.iloc[i]['timestamp'])
#                 flattened_indicators['hour'] = ts.hour
#                 flattened_indicators['minute'] = ts.minute
#                 flattened_indicators['weekday'] = ts.weekday()
#                 flattened_indicators['timestamp'] = ts

#                 if up_j is not None and down_j is not None:
#                     break
    
#             dataset.append(flattened_indicators)

#             candidate_js = [idx for idx in [up_j, down_j] if idx is not None]
#             extreme_j = min(candidate_js) if candidate_js else None

        
#             # Update i for next iteration
#             if extreme_j:
#                 i = extreme_j + 1  # jump to candle after the extreme
#             else:
#                 i += 1  
    
#         start_idx += step_size  # move to next window
#         # Convert to DataFrame
#     df_dataset = pd.DataFrame(dataset)
#     df_dataset.fillna(np.nan, inplace=True)

#     return df_dataset

In [7]:
def flatten_and_align_indicators(ind_dict, N):
    """
    Convert indicators dict to uniform lists of length N.
    Scalars are repeated; lists are padded/truncated to length N.
    Supertrend dicts are flattened.
    Skips 'ema200' indicator.
    """
    aligned = {}
    
    def pad_list(lst, length, fill=np.nan):
        if lst is None: lst = []
        lst = list(lst)
        return [fill]*(length - len(lst)) + lst[-length:] if len(lst) < length else lst[-length:]
    
    for key, val in ind_dict.items():
        if isinstance(val, list):
            if val and isinstance(val[0], dict):
                for subkey in val[0]:
                    aligned[f"{key}_{subkey}"] = pad_list([v[subkey] for v in val], N)
            else:
                aligned[key] = pad_list(val, N)
        else:
            aligned[key] = [val]*N
    return aligned

In [8]:
def list_to_rows(df):
    list_cols = [col for col in [
       'rsi','macd_line','macd_signal','macd_hist','vwap','adx',
       'plus_di','minus_di','stoch_k','stoch_d','boll_upper',
       'boll_middle','boll_lower','atr','obv','supertrend_value',
       'supertrend_trend','ema10','ema20','ema50','ema100','ema200','ema_cross','boll_percent','cci','roc','momentum3'
    ] if col in df.columns]

    scalar_cols = [col for col in df.columns if col not in list_cols]

    # compute max length per row only for existing list columns
    if list_cols:
        max_len_per_row = df[list_cols].applymap(lambda x: len(x) if isinstance(x, list) else 1).max(axis=1)
    else:
        max_len_per_row = pd.Series([1]*len(df))

    def pad_list(lst, length):
        lst = lst if isinstance(lst, list) else [lst]
        return lst + [np.nan]*(length - len(lst))

    # Pad lists and repeat scalars
    for col in df.columns:
        if col in list_cols:
            df[col] = [pad_list(lst, l) for lst, l in zip(df[col], max_len_per_row)]
        else:
            df[col] = [[v]*l for v, l in zip(df[col], max_len_per_row)]

    # Explode all columns
    df_exploded = df.apply(pd.Series.explode).reset_index(drop=True)
    return df_exploded

In [101]:
def append_to_dataset(flattened_indicators, block_df, dataset, distance, extreme_type, pct_change, 
                      current_close, timeframe, high_wick, low_wick, window_size):
    """
    Enrich flattened indicators with additional features and append to dataset.
    Computes time_to_extreme, relative OHLC, volume stats, momentum, and time features.
    """

    # ---------- Time-to-extreme ----------
    flattened_indicators['time_to_extreme'] = distance * timeframe
    flattened_indicators['time_to_extreme_rel'] = distance / window_size  # relative to window length
    flattened_indicators['extreme_type'] = extreme_type
    flattened_indicators['change_percentage'] = pct_change

    # ---------- Wicks ----------
    flattened_indicators['high_wick'] = high_wick
    flattened_indicators['low_wick'] = low_wick

    # ---------- Volume features ----------
    vol = block_df['volume']
    flattened_indicators['volume_sum'] = vol.sum()
    flattened_indicators['volume_avg'] = vol.mean()
    flattened_indicators['volume_rel'] = vol.mean() / vol.iloc[0]
    flattened_indicators['volume_max_rel'] = vol.max() / vol.mean()

    # ---------- High/Low & Range ----------
    flattened_indicators['rel_high'] = block_df['high'].max() / current_close
    flattened_indicators['rel_low'] = block_df['low'].min() / current_close
    flattened_indicators['range_pct'] = (block_df['high'].max() - block_df['low'].min()) / current_close * 100
    flattened_indicators['subblock_len'] = distance

    # ---------- OHLC relative to first candle ----------
    first_candle = block_df.iloc[0]
    flattened_indicators['open_rel'] = first_candle['open'] / current_close
    flattened_indicators['high_rel'] = first_candle['high'] / current_close
    flattened_indicators['low_rel'] = first_candle['low'] / current_close
    flattened_indicators['close_rel'] = first_candle['close'] / current_close

    # ---------- Short-term momentum ----------
    if len(block_df) > 1:
        flattened_indicators['pct_change_1'] = (first_candle['close'] - first_candle['open']) / current_close * 100
        flattened_indicators['pct_change_2'] = (first_candle['close'] - block_df.iloc[1]['close']) / current_close * 100
    else:
        flattened_indicators['pct_change_1'] = 0.0
        flattened_indicators['pct_change_2'] = 0.0

    # ---------- Time features ----------
    ts = pd.to_datetime(first_candle['timestamp'])
    flattened_indicators['hour'] = ts.hour
    flattened_indicators['minute'] = ts.minute
    flattened_indicators['weekday'] = ts.weekday()
    flattened_indicators['timestamp'] = ts

    dataset.append(flattened_indicators)


In [135]:
def create_memecoin_dataset_extremes(df, window_size, step_size, h, threshold_factor, timeframe):
    """
    Create ML dataset using rolling windows with fixed-horizon labeling.

    Features include:
    - OHLC relative
    - Short-term momentum
    - Volume features
    - Time features
    - First-threshold distance & relative distance
    - Wicks from threshold candle

    Labels:
    - extreme_type: 'up', 'down', 'stable'
    - change_percentage: % move at first threshold or at horizon for stable
    """
    dataset = []
    N = len(df)
    start_idx = 0

    while start_idx + h <= N:
        block_df = df.iloc[start_idx:start_idx + window_size].copy()
        current_close = block_df.iloc[0]['close']

        # ---------- Fixed Horizon Labels with first-threshold detection ----------
        sub_closes = df['close'].iloc[start_idx:start_idx + h+1].values
        extreme_type = 'stable'
        distance = h
        pct_change = (sub_closes[-1] - current_close) / current_close * 100.0  # default horizon change

        threshold_idx = start_idx + h  # default: last candle in horizon

        for i, future_close in enumerate(sub_closes[1:], start=1):
            pct = (future_close - current_close) / current_close * 100.0
            if pct >= threshold_factor:
                extreme_type = 'up'
                distance = i
                pct_change = pct
                threshold_idx = start_idx + i
                break
            elif pct <= -threshold_factor:
                extreme_type = 'down'
                distance = i
                pct_change = pct
                threshold_idx = start_idx + i
                break

        distance = min(distance, h)
        
        # ---------- Wicks ----------
        high_wick = df.iloc[threshold_idx]['high']
        low_wick = df.iloc[threshold_idx]['low']

        # ---------- Compute indicators ----------
        indicators = get_indicators_for_token(block_df.to_dict('records'))
        flattened_indicators = flatten_and_align_indicators(indicators, len(block_df))

        # ---------- Append to dataset ----------
        append_to_dataset(
            flattened_indicators,
            block_df,
            dataset,
            distance,
            extreme_type,
            pct_change,
            current_close,
            timeframe,
            high_wick,
            low_wick,
            window_size
        )

        start_idx += step_size

    df_dataset = pd.DataFrame(dataset)
    df_dataset.fillna(np.nan, inplace=True)
    return df_dataset


In [136]:
dataset = create_memecoin_dataset_extremes(df, window_size=300, step_size = 50, h=100, threshold_factor=15.0, timeframe=5)

# dataset = create_forward_label_dataset(df, window_size=1000, step_size = 100, up_threshold=15, down_threshold=10, stable_threshold = 5, 
#                                       timeframe=5, max_horizon=300)

exploded_dataset = list_to_rows(dataset)

In [137]:
exploded_dataset.shape

(20800, 51)

In [138]:
exploded_dataset.columns

Index(['rsi', 'ema10', 'ema20', 'ema50', 'ema100', 'ema200', 'ema_cross',
       'macd_line', 'macd_signal', 'macd_hist', 'vwap', 'adx', 'plus_di',
       'minus_di', 'stoch_k', 'stoch_d', 'boll_upper', 'boll_middle',
       'boll_lower', 'boll_percent', 'atr', 'obv', 'supertrend_value',
       'supertrend_trend', 'cci', 'roc', 'momentum3', 'time_to_extreme',
       'time_to_extreme_rel', 'extreme_type', 'change_percentage', 'high_wick',
       'low_wick', 'volume_sum', 'volume_avg', 'volume_rel', 'volume_max_rel',
       'rel_high', 'rel_low', 'range_pct', 'subblock_len', 'open_rel',
       'high_rel', 'low_rel', 'close_rel', 'pct_change_1', 'pct_change_2',
       'hour', 'minute', 'weekday', 'timestamp'],
      dtype='object')

In [139]:
exploded_dataset_clean = exploded_dataset.dropna(how="any").reset_index(drop=True)

In [140]:
counts = exploded_dataset_clean['extreme_type'].value_counts()
# counts2 = exploded_dataset_clean['fixed_h_direction'].value_counts()
print(counts)
# print(counts2)

up        8897
stable    6264
down      4492
Name: extreme_type, dtype: int64


In [141]:
exploded_dataset_clean.shape

(19653, 51)

In [142]:
exploded_dataset_clean['time_to_extreme'].describe()

count     19653
unique       41
top         500
freq       6264
Name: time_to_extreme, dtype: int64

In [88]:

def balance_memecoin_dataset(df, random_state=42):
    """
    Downsample the majority 'stable' class in both extreme_type and fixed_h_direction
    to create a more balanced dataset for ML.
    
    Parameters:
        df : pd.DataFrame
            Original dataset with 'extreme_type' and 'fixed_h_direction' columns
        random_state : int
            Seed for reproducibility
    
    Returns:
        df_balanced : pd.DataFrame
            Balanced dataset
    """

    # -----------------------
    # Step 1: Balance extreme_type
    # -----------------------
    up_down_rows = df[df['extreme_type'].isin(['up','down'])]
    stable_rows_extreme = df[df['extreme_type'] == 'stable']

    # Target stable count = average of up/down
    target_stable_count_extreme = int(len(up_down_rows) / 2)
    stable_sampled_extreme = stable_rows_extreme.sample(n=target_stable_count_extreme,
                                                        random_state=random_state)

    df_extreme_balanced = pd.concat([up_down_rows, stable_sampled_extreme])

    return df_extreme_balanced


In [90]:
df_new = balance_memecoin_dataset(exploded_dataset_clean)

In [92]:
counts = df_new['extreme_type'].value_counts()
print(counts)

In [94]:
df_new.shape

(7332, 36)

In [95]:
df_new['time_to_extreme'].describe()

count     7332
unique      39
top       2985
freq      2370
Name: time_to_extreme, dtype: int64

In [41]:
exploded_dataset_clean.head(10)

,rsi,ema10,ema20,ema50,ema100,ema200,ema_cross,macd_line,macd_signal,macd_hist,...,open_rel,high_rel,low_rel,close_rel,pct_change_1,pct_change_2,hour,minute,weekday,timestamp
0,68.77,310011.8,297558.61,272711.21,246126.72,218765.1,1,14367.84,8591.74,5776.1,...,0.995911,1.0,0.999677,1.0,0.40894,0.488094,5,50,6,2025-08-17 05:50:40
1,68.77,310011.8,297558.61,272711.21,246126.72,218765.1,1,14367.84,8591.74,5776.1,...,0.995911,1.0,0.999677,1.0,0.40894,0.488094,5,50,6,2025-08-17 05:50:40
2,68.77,310011.8,297558.61,272711.21,246126.72,218765.1,1,14367.84,8591.74,5776.1,...,0.995911,1.0,0.999677,1.0,0.40894,0.488094,5,50,6,2025-08-17 05:50:40
3,68.77,310011.8,297558.61,272711.21,246126.72,218765.1,1,14367.84,8591.74,5776.1,...,0.995911,1.0,0.999677,1.0,0.40894,0.488094,5,50,6,2025-08-17 05:50:40
4,68.77,310011.8,297558.61,272711.21,246126.72,218765.1,1,14367.84,8591.74,5776.1,...,0.995911,1.0,0.999677,1.0,0.40894,0.488094,5,50,6,2025-08-17 05:50:40
5,68.77,310011.8,297558.61,272711.21,246126.72,218765.1,1,14367.84,8591.74,5776.1,...,0.995911,1.0,0.999677,1.0,0.40894,0.488094,5,50,6,2025-08-17 05:50:40
6,68.77,310011.8,297558.61,272711.21,246126.72,218765.1,1,14367.84,8591.74,5776.1,...,0.995911,1.0,0.999677,1.0,0.40894,0.488094,5,50,6,2025-08-17 05:50:40
7,68.77,310011.8,297558.61,272711.21,246126.72,218765.1,1,14367.84,8591.74,5776.1,...,0.995911,1.0,0.999677,1.0,0.40894,0.488094,5,50,6,2025-08-17 05:50:40
8,68.77,310011.8,297558.61,272711.21,246126.72,218765.1,1,14367.84,8591.74,5776.1,...,0.995911,1.0,0.999677,1.0,0.40894,0.488094,5,50,6,2025-08-17 05:50:40
9,68.77,310011.8,297558.61,272711.21,246126.72,218765.1,1,14367.84,8591.74,5776.1,...,0.995911,1.0,0.999677,1.0,0.40894,0.488094,5,50,6,2025-08-17 05:50:40


In [32]:
with pd.ExcelWriter(r"C:\Users\vamsh\Downloads\TA MV2\python_server\ML_Training_datasets\memecoin_training_dataset.xlsx") as writer:
    exploded_dataset_clean.to_excel(writer, sheet_name="5s_candles")

In [54]:

folder_path = r"C:\Users\vamsh\Downloads\TA MV2\python_server\ML_Training_datasets\CandleData"
base_excel_path = r"C:\Users\vamsh\Downloads\TA MV2\python_server\ML_Training_datasets\memecoin_training_dataset_5S.xlsx"
sheet_name = "5s_candles"

json_files = [f for f in os.listdir(folder_path) if f.endswith(".json")]
batch_size = 5  # create a new workbook every 5 JSONs
file_number = 1

created_excel_files = []

for batch_idx, i in enumerate(range(0, len(json_files), batch_size)):
    batch_files = json_files[i:i+batch_size]

    # Name for this batch workbook
    excel_file_path = base_excel_path.replace(".xlsx", f"_{batch_idx+1}.xlsx")

    # Create new workbook for this batch
    book = Workbook()
    default_sheet = book.active
    if default_sheet.title != sheet_name:
        book.remove(default_sheet)
    book.create_sheet(sheet_name)
    book.save(excel_file_path)
    created_excel_files.append(excel_file_path)

    startrow = 0

    for file_name in batch_files:
        file_path = os.path.join(folder_path, file_name)
        with open(file_path, "r") as f:
            data = json.load(f)

        token_name = data.get("name", "Unknown")
        candles = data.get("timeframes", {}).get("15S", [])

        if not candles:
            print(f"No candles found in {file_name}. Skipping.")
            continue

        df = pd.DataFrame(candles)
        if "timestamp" in df.columns:
            df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        
        # Sort by timestamp just in case
        df = df.sort_values("timestamp").reset_index(drop=True)

        print(f"{file_number}, Processing token: {token_name}, file: {file_name}")

        dataset = create_memecoin_dataset_extremes(df, window_size=500, step_size = 50, min_block=200, h=100, threshold_factor=20.0, timeframe=15)
        exploded_dataset = list_to_rows(dataset)
        exploded_dataset_clean = exploded_dataset.dropna(how="any").reset_index(drop=True)

        # Append to Excel
        with pd.ExcelWriter(excel_file_path, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
            exploded_dataset_clean.to_excel(
                writer,
                sheet_name=sheet_name,
                startrow=startrow,
                header=(startrow == 0),
                index=False
            )

        startrow += len(exploded_dataset_clean)
        file_number += 1


required_columns = ['rsi', 'ema10', 'ema20', 'ema50', 'ema100', 'ema200', 'ema_cross',
       'macd_line', 'macd_signal', 'macd_hist', 'vwap', 'adx', 'plus_di',
       'minus_di', 'stoch_k', 'stoch_d', 'boll_upper', 'boll_middle',
       'boll_lower', 'boll_percent', 'atr', 'obv', 'supertrend_value',
       'supertrend_trend', 'cci', 'roc', 'momentum3', 'extreme_type',
       'time_to_extreme', 'change_percentage', 'volume_sum', 'volume_avg',
       'volume_rel', 'volume_max_rel', 'fixed_h_pct_change',
       'fixed_h_direction', 'rel_high', 'rel_low', 'range_pct', 'subblock_len',
       'open_rel', 'high_rel', 'low_rel', 'close_rel', 'pct_change_1',
       'pct_change_2', 'hour', 'minute', 'weekday', 'timestamp']

# Read and concatenate all files
df_list = [pd.read_excel(file) for file in created_excel_files]
merged_df = pd.concat(df_list, ignore_index=True)

merged_df = merged_df.dropna(subset=required_columns)

# Save to a new Excel file
merged_df.to_excel(r"C:\Users\vamsh\Downloads\TA MV2\python_server\ML_Training_datasets\5S_400Window_100Step_100h_10th.xlsx", index=False)
print("Files merged successfully!")

1, Processing token: Rizzoween, file: 2dCYTTCQxPsMZeBR2b3x3wmKYytfxsnsEtyy7nwGQzEU.json
2, Processing token: PHI, file: 2RKZJaD7sSues6YxoujsafcBYjxW6t4h9AVgiRWyDjdw.json
3, Processing token: AIONIX, file: 2XKwu8fJTrV22UjH8jThejxtctHDCspiubdEmbx2A4dv.json
4, Processing token: Sus, file: 32jnad7jYQdLcap3kcKtX4NwetT8P1yc8c2SxfbReUqh.json
5, Processing token: SPARK, file: 3mgxf9VVDd82E75bovuAJPS1wYmJoZCAo4MB6zpQDjnv.json
6, Processing token: PUMPLESS, file: 4yXG1V68A7BU1vKbBMACDBbBhbp3P5p5uppRZkyWubuV.json
7, Processing token: snail, file: 5QcEDqaiTMbTmdzhPueEQQE6REomd7efGE4Wd3LsJT8j.json
8, Processing token: SOS, file: 5vvWcHHo3drWnhYqwhum6aNnkxc8JQaycaCRXto7AXpC.json
9, Processing token: SOL6900, file: 6xb5dZ6z4jSrCWeUMvky6xquA1wAdavR9sizV4UtEdHJ.json
10, Processing token: TH3000, file: 72aBJ5Py4a6XXCtPBDhwubcZwv76xKqt8ws9zPupVjMd.json
11, Processing token: glaze, file: 99JmBTZSsEBSmVBKP1jeSKfU5DAvMdHPnssziKNzHgBa.json
12, Processing token: Lenny, file: 9MXxzYYszFBu6CvN57hkRrgKXFHGk5PyTL

C:\Users\vamsh\AppData\Local\Temp\ipykernel_39164\545696013.py:15: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  max_len_per_row = pd.Series([1]*len(df))


16, Processing token: BoatKid, file: AjpHgYAM8BK5cJo2knzATeTWB5Yk8sVqWNJVjbNRTRRN.json
17, Processing token: Reachy, file: BkqpKKSS2ia6ASUgjBEJTsPHWkfA9dmV17zADq3rHmkr.json
18, Processing token: iOQ, file: BMBcZ9GWMCi9HaCE7BagrLxakzffy6fAGdEpihLRfVPw.json
19, Processing token: DIRECTED, file: DZTd1F3SGRGHGyJnq1huzXyPqHenHFMYc5y8MRea5Auz.json
20, Processing token: Halo, file: E2at1duW2nznvYzh6seUD4X3e3zw2Mjq9PeeusvrtrYH.json
21, Processing token: FORK, file: E9pd39bsqnQ62FVB4o7HMm8RZ8Hw2zG5wNsZDVgfBcJJ.json
22, Processing token: SLM, file: FCLENzDCvwSBedRJn8mHvY1fAMKTyrWGYoJjGfJMy7v9.json
23, Processing token: SP500, file: FkbRy6LErYZcLoGXcMtAGnCVVpRVdGeDUaJPaxB6cSDe.json
24, Processing token: CLIPPY, file: HJtdALk2oebbBJibixu6aFCoKUjQFUZjKDFvjoYZPxUa.json
25, Processing token: reterd, file: HnBgEaBEB27q5g8w7exirYEG8FPCoZunzNAPAJ2ys4Tk.json
26, Processing token: gif, file: HztpMKvrMD5r4pGge6b1GvenPejVye5hBjCtwKxXj99K.json
27, Processing token: Tika, file: qASXryCs5Vz5XYf2BhgNqchAicFjp7n

C:\Users\vamsh\AppData\Local\Temp\ipykernel_39164\545696013.py:15: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  max_len_per_row = pd.Series([1]*len(df))


28, Processing token: FARTLESS, file: UKSPrYDU4veB4cAfV9HkzNg9gNP1jsoch8oySMG9tcJ.json
Files merged successfully!
